# Plexus — Step-2 Reachability Notebook

**Supersedes the Step-1 Gate-3 model.** Step-1 proved the structure holds (clustering PASS, stratification PASS). Its Gate-3 *bridge* model failed validation against its own Gate-1: it used **asymmetric top-K coverage** for reachability while Gate-1 used **symmetric full-vector cosine** for clustering — two different instruments — and the coverage metric manufactured 7 "islands" where the cosine graph had only 2 isolated nodes at the same threshold (a 3.5× inflation). The islands were largely a metric artifact, not a data fact.

**This notebook rebuilds reachability the right way. Build order (agreed with the independent review):**
- **Fix A — symmetric cosine.** Reachability uses the *same* full-vector cosine that Gate-1 validated. No top-K coverage anywhere.
- **Fix E — cleaned skill normalisation.** Junk tokens (`senior`, `it services`, `dot`, `technical support`…) are blocklisted at parse time; `.net` / `c#` / `c++` are preserved as exact tokens. **Run the Gate-1 re-check (Diagnostic 1) and eyeball it before trusting anything downstream** — cleaning changes the cosine distribution, so we confirm the validated connections survive cleaning first.
- **Fix B — node-level stratification.** `QA @ Services` and `QA @ GCC` are distinct vertices. Reachability is computed on the stratified graph, but the **default view is within-stratum** (a Services worker sees Services doors). **Cross-stratum is a separate, labelled overlay**, never mixed into the default — this preserves the moat and prevents the "recommend Kubernetes to a Services-Java dev because a GCC node pulled it in" failure.
- **Fix C — two-hop Pathfinder, explicit intermediate.** Every role gets a *"you are here → named door → what it opens"* readout. Connected roles show reach-expansion; sparse roles show the nearest door + the skills to bridge to it. **Same-stratum intermediates in the default view; cross-stratum stepping-stones only in the labelled overlay.** No role is labelled "island" — every role gets a path of some length.

**Deferred (deliberately):** adaptive-K, directional asymmetry, lower thresholds (the last is the honesty-laundering path — off unless A/C leave a real problem).

**Run-and-see, not predicted:** we do NOT assume how many of Step-1's 7 islands reconnect. Gate-1 data only supports reconnection for QA, DBA, Data-Scientist/ML. Security and Network/Sys-Admin may be genuinely sparse — the notebook will tell us.


## 0 · Config & imports

Edit `SUBSTRATE` / `GCC_FILE` to your local paths. Standard deps only (pandas, numpy, scikit-learn, networkx).

In [1]:
import re
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx

pd.set_option("display.max_colwidth", 60)

# --- paths ---
SUBSTRATE = "indian-job-market-dataset-2025.xlsx"
GCC_FILE  = "GCC-Journal-India-List.xlsx"

# --- tunables ---
MIN_ROLE_POSTINGS = 80
MIN_STRATUM       = 25
CORE_K            = 12
EDGE_THRESHOLDS   = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
GRAPH_THRESHOLD   = 0.25   # confirmed from sweep: 0 isolated @ 0.25 (0.30 reintroduces 2; 0.15-0.20 = hairball)

# Fix E — parse-time NOISE blocklist: tokens that are NOT skills and leaked into every vector
NOISE = {'senior','junior','it services','dot','technical support','web services','good','communication',
 'good communication','years','experience','responsible','ability','work','team','it','company',
 'role','job','requirement','knowledge','strong','excellent','skills','skill','plus','etc','various',
 'new','using','related','based','per','will','must','candidate','position','immediate joiner','notice period'}

# alias map (extended for .net family so it stops fragmenting into 'dot'/'net')
ALIAS = {'reactjs':'react','react.js':'react','react js':'react','nodejs':'node','node.js':'node','node js':'node',
 'js':'javascript','ts':'typescript','k8s':'kubernetes','aws cloud':'aws','amazon web services':'aws',
 'mssql':'sql server','ms sql':'sql server','postgresql':'postgres','mongo':'mongodb',
 'dot net':'.net','dotnet':'.net','dot net core':'.net core','asp.net':'.net','asp net':'.net',
 'c sharp':'c#','csharp':'c#','springboot':'spring boot','restful':'rest api','rest':'rest api',
 'cicd':'ci cd','ci/cd':'ci cd','ml':'machine learning','powerbi':'power bi','html5':'html','css3':'css'}

# SKILL_STOP — generic-but-real skills stripped when building DISCRIMINATIVE role cores (not bridges)
SKILL_STOP = {'software','development','software development','software engineering','programming','coding',
 'technical','troubleshooting','analytical','problem solving','teamwork','sales','administration',
 'management','design','testing','engineering','operations','support','documentation','agile','scrum',
 'leadership','time management','information technology','computer science','project management',
 'application development','development methodologies','business intelligence'}


## 1 · Load & confirm shapes

In [2]:
df  = pd.read_excel(SUBSTRATE, sheet_name="Sheet1")
gcc = pd.read_excel(GCC_FILE, sheet_name="All GCCs \u2013 India Master")
print("Substrate:", df.shape, "| GCC master:", gcc.shape)
assert {"title","tagsAndSkills","companyName"} <= set(df.columns)
assert "Company / GCC Name" in gcc.columns


Substrate: (97929, 17) | GCC master: (274, 9)


## 2 · Step 1 — IT subset filter (settled in Step-1)

In [3]:
TECH_TITLE = ['software','developer','programmer','full stack','fullstack','frontend','front end','backend',
'back end','devops',' sre','data scien','data engineer','data analyst','ml engineer','machine learning',' ai ',
'ai engineer','qa ','sdet','test engineer','automation test','tester','web develop','mobile develop','android',
'ios develop','java develop','python develop','dot net','.net','php develop','cloud engineer','cloud architect',
'solution architect','technical architect','database admin','dba','system admin','network engineer',
'security engineer','cyber','sap ','salesforce','tech lead','engineering manager','data architect','etl',
'bi develop','platform engineer']
TECH_SKILL = ['java','python','javascript','typescript','react','angular','vue','node','express','aws','azure',
'gcp','kubernetes','docker','sql','mysql','postgres','mongodb','spring','hibernate','c++','c#','.net','php',
'html','css','git','jenkins','kafka','spark','hadoop','tensorflow','pytorch','selenium','rest api',
'microservices','devops','linux','terraform','redis','elasticsearch','scala','golang','rust']
t = df["title"].fillna("").str.lower(); s = df["tagsAndSkills"].fillna("").str.lower()
is_it = t.apply(lambda x: any(k in x for k in TECH_TITLE)) | s.apply(lambda x: sum(1 for k in TECH_SKILL if k in x) >= 2)
sub = df[is_it].copy().reset_index(drop=True)
print(f"IT subset: {len(sub)} ({len(sub)/len(df)*100:.1f}%)")


IT subset: 27700 (28.3%)


## 3 · Step 2 — CLEANED skill normalisation (Fix E)

NOISE tokens dropped at parse; `.net`/`c#`/`c++` preserved as exact tokens. The printout confirms `senior` / `it services` are gone from the skill space (they were in 21/21 role vectors in Step-1, inflating every cosine).

In [4]:
def clean_tok(tok):
    tok = tok.strip().lower()
    tok = re.sub(r'[^a-z0-9+#. /]', ' ', tok)   # keep + # . so c#, c++, .net survive
    tok = re.sub(r'\s+', ' ', tok).strip()
    return ALIAS.get(tok, tok)

def parse_skills(cell):
    if not isinstance(cell, str): return []
    out = []
    for raw in cell.split(','):
        c = clean_tok(raw)
        if c and len(c) > 1 and c not in NOISE:
            out.append(c)
    return out

sub["skills"] = sub["tagsAndSkills"].apply(parse_skills)
allsk = Counter(sk for lst in sub["skills"] for sk in lst)
print("avg skills/posting:", round(sub['skills'].apply(len).mean(),1))
print("\nTOP 25 cleaned skills:")
for sk,c in allsk.most_common(25): print(f"  {sk:26s} {c}")
print("\n[check] junk tokens gone?",
      {j: allsk.get(j,0) for j in ['senior','it services','dot','technical support','web services']})
print("[check] tech tokens preserved?",
      {j: allsk.get(j,0) for j in ['.net','c#','c++','.net core']})


avg skills/posting: 7.7

TOP 25 cleaned skills:
  python                     5322
  css                        4080
  java                       3977
  sql                        3380
  sap                        3028
  javascript                 3022
  c#                         2586
  development                2096
  rest api                   1968
  aws                        1935
  c++                        1696
  html                       1577
  continuous integration     1566
  react                      1556
  spring boot                1540
  microservices              1535
  kubernetes                 1471
  agile                      1402
  application development    1393
  software development       1384
  software testing           1302
  project management         1201
  oracle                     1169
  automation                 1166
  .net                       1157

[check] junk tokens gone? {'senior': 0, 'it services': 0, 'dot': 0, 'technical support': 0, 'web serv

## 4 · Canonical roles + de-hub (settled in Step-1)

In [5]:
CANON = [
    ('Data Scientist / ML',        ['data scientist','machine learning','ml engineer','ai engineer','deep learning','data science']),
    ('Data Engineer',              ['data engineer','etl','big data','hadoop','spark develop','databricks']),
    ('Data / BI Analyst',          ['data analyst','business analyst','bi develop','bi analyst','power bi','tableau','analytics']),
    ('DevOps / SRE',               ['devops','site reliability',' sre','platform engineer','build and release','cloudops']),
    ('Cloud Engineer / Architect', ['cloud engineer','cloud architect','aws engineer','azure engineer','cloud consultant']),
    ('Security Engineer',          ['security engineer','cyber','infosec','soc analyst','penetration','vapt']),
    ('QA / Test Engineer',         ['qa','quality assurance','test engineer','sdet','automation test','software tester','testing','test analyst']),
    ('Full Stack Developer',       ['full stack','fullstack','mean stack','mern','mern stack']),
    ('Frontend Developer',         ['frontend','front end','react develop','angular develop','ui developer','ui engineer','ui/ux develop']),
    ('Mobile Developer',           ['android develop','ios develop','mobile develop','flutter','react native develop']),
    ('.NET Developer',             ['.net','dot net','c# develop','asp.net']),
    ('PHP / Web Developer',        ['php develop','laravel','wordpress developer','codeigniter','core php']),
    ('Backend / Java Developer',   ['backend','back end','java develop','spring develop','node develop','python develop','golang develop','api develop']),
    ('Database Architect',         ['database architect','data architect']),
    ('Database Administrator',     ['dba','database admin','oracle dba','sql dba','database engineer']),
    ('Solution / Tech Architect',  ['solution architect','technical architect','software architect','enterprise architect']),
    ('Eng Manager / Tech Lead',    ['engineering manager','tech lead','technical lead','team lead','development lead','delivery lead']),
    ('SAP Consultant',             ['sap']),
    ('Salesforce Developer',       ['salesforce']),
    ('Network / Sys Admin',        ['network engineer','system administrator','system admin','network admin','windows admin','linux admin','noc']),
    ('Software Engineer (General)',['software engineer','software developer','programmer','sde','application developer','it developer']),
]
def assign_role(title):
    tl = str(title).lower()
    for lab,kw in CANON:
        if any(k in tl for k in kw): return lab
    return 'unclassified'
sub["role"] = sub["title"].apply(assign_role)

SIGNALS = {
 'Frontend Developer':       ['react','angular','vue','javascript','typescript','redux','rxjs','html','css','responsive web design','frontend development','next js'],
 'Backend / Java Developer': ['spring','spring boot','hibernate','microservices','j2ee','jpa','rest api','spring mvc','kafka'],
 '.NET Developer':           ['.net','.net core','c#','asp','mvc','entity framework','wpf'],
 'PHP / Web Developer':      ['php','laravel','wordpress','codeigniter','magento'],
 'Data Scientist / ML':      ['machine learning','deep learning','tensorflow','pytorch','keras','nlp','computer vision','data scientist','pandas'],
 'Data Engineer':            ['etl','spark','hadoop','pyspark','hive','airflow','snowflake','data warehousing','scala','databricks'],
 'Data / BI Analyst':        ['power bi','tableau','data analysis','data visualization','qlik'],
 'DevOps / SRE':             ['devops','kubernetes','docker','terraform','jenkins','ci cd','ansible','sre','aws devops'],
 'QA / Test Engineer':       ['selenium','automation testing','sdet','appium','cypress','playwright','rest assured','test automation','manual testing'],
 'Database Administrator':   ['oracle dba','rman','oracle rac','database administration','sql server dba'],
 'SAP Consultant':           ['sap','sap abap','sap sd','sap fico','sap hana','sap mm'],
 'Salesforce Developer':     ['salesforce','apex','visualforce','soql','salesforce lightning'],
 'Mobile Developer':         ['android','ios','flutter','kotlin','swift','react native'],
 'Security Engineer':        ['penetration testing','vapt','siem','soc','cyber security','owasp'],
}
def reroute(skills):
    sk=set(skills); best=None;bn=0;second=0
    for role,sig in SIGNALS.items():
        h=sum(1 for x in sig if x in sk)
        if h>bn: second=bn; bn=h; best=role
        elif h>second: second=h
    return best if (bn>=2 and bn-second>=1) else None
mask = sub["role"]=="Software Engineer (General)"
rr = sub.loc[mask,"skills"].apply(reroute)
sub.loc[mask,"role"] = rr.where(rr.notna(),"Software Engineer (General)")
print(f"de-hub rerouted {int(rr.notna().sum())} postings out of Software Engineer (General)")

role_counts = sub["role"].value_counts()
node_roles = [r for r,c in role_counts.items() if r!='unclassified' and c>=MIN_ROLE_POSTINGS]
print(f"graph-eligible role nodes: {len(node_roles)}")


de-hub rerouted 1201 postings out of Software Engineer (General)
graph-eligible role nodes: 21


## 5 · Role vectors on CLEANED skills (exact-token TF-IDF)

Custom analyzer = identity, so `c#` / `.net` / multiword skills survive intact (Step-1's default tokenizer silently dropped them). TF-IDF still down-weights ubiquitous skills — correct for *clustering*. Reachability (Fix A) will use these same full vectors via cosine.

In [6]:
def core(role, k=CORE_K, stratum=None):
    m = (sub.role==role)
    if stratum is not None: m &= (sub.emp_type==stratum)
    toks = [s for lst in sub.loc[m,"skills"] for s in lst if s not in SKILL_STOP]
    return [w for w,_ in Counter(toks).most_common(k)]

role_docs = {}
for r in node_roles:
    role_docs[r] = [s for lst in sub.loc[sub.role==r,"skills"] for s in lst]

roles = list(role_docs.keys())
vec = TfidfVectorizer(analyzer=lambda x: x, min_df=2, sublinear_tf=True)
X = vec.fit_transform([role_docs[r] for r in roles])
SIM = cosine_similarity(X)
print("role-vector matrix:", X.shape)


role-vector matrix: (21, 3554)


## ✅ Diagnostic 1 — Gate-1 nearest neighbours on CLEANED vectors (CHECKPOINT)

Do the Step-1-validated connections survive cleaning? Compare these to Step-1's numbers (QA→SE was 0.40, Backend→Full Stack 0.63). If cleaning shifted them materially, **stop and inspect before trusting reachability** — the whole Fix-A argument rests on these connections being real.

In [7]:
def nn(role, k=5):
    i = roles.index(role)
    order = SIM[i].argsort()[::-1]
    return [(roles[j], round(float(SIM[i][j]),2)) for j in order if roles[j]!=role][:k]

for p in ['QA / Test Engineer','Backend / Java Developer','Database Administrator',
          'Data Scientist / ML','DevOps / SRE','Frontend Developer','Security Engineer']:
    if p in roles: print(f"{p:28s} -> {nn(p)}")


QA / Test Engineer           -> [('Software Engineer (General)', 0.4), ('Backend / Java Developer', 0.33), ('Eng Manager / Tech Lead', 0.32), ('DevOps / SRE', 0.3), ('Full Stack Developer', 0.29)]
Backend / Java Developer     -> [('Full Stack Developer', 0.63), ('Frontend Developer', 0.43), ('Software Engineer (General)', 0.43), ('DevOps / SRE', 0.42), ('Eng Manager / Tech Lead', 0.41)]
Database Administrator       -> [('DevOps / SRE', 0.27), ('Data Engineer', 0.25), ('Software Engineer (General)', 0.23), ('Database Architect', 0.21), ('Backend / Java Developer', 0.2)]
Data Scientist / ML          -> [('Data Engineer', 0.37), ('Data / BI Analyst', 0.36), ('Backend / Java Developer', 0.33), ('Software Engineer (General)', 0.32), ('Solution / Tech Architect', 0.31)]
DevOps / SRE                 -> [('Cloud Engineer / Architect', 0.49), ('Backend / Java Developer', 0.42), ('Full Stack Developer', 0.4), ('Software Engineer (General)', 0.39), ('Data Engineer', 0.38)]
Frontend Developer     

## 6 · Fix A — symmetric-cosine reachability + threshold sweep (full role graph)

Reachability = cosine ≥ threshold, the same metric Gate-1 validated. **This directly answers the islands question:** how many roles are isolated under the correct metric (Step-1's coverage model found 7)? Pick `GRAPH_THRESHOLD` from this sweep — cleaning changed the distribution, so don't inherit 0.30 blindly.

In [8]:
print(f"{'thr':>5} {'edges':>6} {'avg_deg':>8} {'components':>11} {'isolated':>9}")
for thr in EDGE_THRESHOLDS:
    G=nx.Graph(); G.add_nodes_from(roles)
    for i in range(len(roles)):
        for j in range(i+1,len(roles)):
            if SIM[i][j]>=thr: G.add_edge(roles[i],roles[j])
    degs=[d for _,d in G.degree()]
    print(f"{thr:>5} {G.number_of_edges():>6} {np.mean(degs):>8.2f} {nx.number_connected_components(G):>11} {sum(1 for d in degs if d==0):>9}")

G = nx.Graph(); G.add_nodes_from(roles)
for i in range(len(roles)):
    for j in range(i+1,len(roles)):
        if SIM[i][j]>=GRAPH_THRESHOLD: G.add_edge(roles[i],roles[j],weight=round(float(SIM[i][j]),3))
isolated = [r for r in roles if G.degree(r)==0]
print(f"\n@ GRAPH_THRESHOLD={GRAPH_THRESHOLD}: {G.number_of_edges()} edges, isolated nodes ({len(isolated)}): {isolated}")
print(">> compare to Step-1 Gate-3 coverage model: 7 islands. How many survive the correct metric?")


  thr  edges  avg_deg  components  isolated
 0.15    157    14.95           1         0
  0.2    118    11.24           1         0
 0.25     85     8.10           1         0
  0.3     49     4.67           4         2
 0.35     31     2.95           7         6
  0.4     14     1.33          12        10

@ GRAPH_THRESHOLD=0.25: 85 edges, isolated nodes (0): []
>> compare to Step-1 Gate-3 coverage model: 7 islands. How many survive the correct metric?


## 7 · Fix B — node-level stratification (within-stratum default + cross-stratum overlay)

`QA @ Services` and `QA @ GCC` are distinct vertices, vectorised over the same vocabulary so cosines are comparable. **Default reachability is within-stratum**; the **cross-stratum set is returned separately and labelled** (it may require credential / brand barriers, not just skills). Only roles with ≥ `MIN_STRATUM` postings in a stratum get a node there.

In [9]:
# employer tag (services / gcc) — token-boundary blocklist FIRST, then genuine GCC
def norm(x): x=str(x).lower(); x=re.sub(r'[^a-z0-9 ]',' ',x); return re.sub(r'\s+',' ',x).strip()
def tokm(n,p): p=norm(p); return bool(p) and re.search(r'(?:^| )'+re.escape(p)+r'(?:$| )',n) is not None
BLOCK=['accenture','wipro','capgemini','cognizant','hcltech','hcl technologies','tcs','tata consultancy',
'tech mahindra','mphasis','hexaware','zensar','mindtree','ltimindtree','ltts','persistent','coforge',
'birlasoft','cybage','conduent','genpact','wns','firstsource','ey','deloitte','kpmg','pwc','ibm','dxc',
'atos','infosys','info edge','nagarro','happiest minds','sonata','cyient','kpit','zoho']
def gcc_brand(raw):
    x=norm(raw)
    for j in ['india','bangalore','bengaluru','hyderabad','pune','mumbai','chennai','delhi','ncr','gurgaon',
              'gurugram','kolkata','gcc','global','capability','centre','center','technology','technologies',
              'solutions','services','engineering','advanced','development']:
        x=re.sub(r'(?:^| )'+j+r'(?:$| )',' ',' '+x+' ').strip()
    return re.sub(r'\s+',' ',x).strip()
genuine=sorted({gcc_brand(r) for r in gcc['Company / GCC Name'].dropna()
                if not any(tokm(norm(r),b) for b in BLOCK) and len(gcc_brand(r))>=3})
def tag(cn):
    nm=norm(cn)
    if any(tokm(nm,b) for b in BLOCK): return 'services'
    if any(tokm(nm,g) for g in genuine): return 'gcc'
    return 'other'
sub['emp_type']=sub['companyName'].apply(tag)

# stratified nodes
strat_docs={}
for r in node_roles:
    for et in ['services','gcc']:
        d=[s for lst in sub.loc[(sub.role==r)&(sub.emp_type==et),"skills"] for s in lst]
        if (sub[(sub.role==r)&(sub.emp_type==et)]).shape[0] >= MIN_STRATUM:
            strat_docs[f"{r} @ {et}"]=d
snodes=list(strat_docs)
svec=TfidfVectorizer(analyzer=lambda x:x, min_df=2, sublinear_tf=True)
SX=svec.fit_transform([strat_docs[n] for n in snodes])
SSIM=cosine_similarity(SX)
def s_strat(n): return n.rsplit(" @ ",1)[1]
def s_role(n):  return n.rsplit(" @ ",1)[0]

def reach_stratified(node, thr):
    i=snodes.index(node); st=s_strat(node)
    within, cross = [], []
    for j,m in enumerate(snodes):
        if j==i or SSIM[i][j]<thr: continue
        (within if s_strat(m)==st else cross).append((s_role(m), round(float(SSIM[i][j]),2)))
    return sorted(within,key=lambda x:-x[1]), sorted(cross,key=lambda x:-x[1])

print(f"stratified nodes ({len(snodes)}): "
      f"{sum(1 for n in snodes if s_strat(n)=='services')} services / {sum(1 for n in snodes if s_strat(n)=='gcc')} gcc\n")
for n in [x for x in snodes if s_strat(x)=='services']:
    w,c = reach_stratified(n, GRAPH_THRESHOLD)
    print(f"[{n}]")
    print(f"   within-stratum doors: {w if w else '(none direct)'}")
    print(f"   cross-stratum (GCC, labelled — may need credential/brand barriers): {c if c else '(none)'}")


stratified nodes (31): 18 services / 13 gcc

[SAP Consultant @ services]
   within-stratum doors: [('Software Engineer (General)', 0.53), ('Eng Manager / Tech Lead', 0.26), ('Solution / Tech Architect', 0.26)]
   cross-stratum (GCC, labelled — may need credential/brand barriers): [('SAP Consultant', 0.47), ('Solution / Tech Architect', 0.27)]
[Software Engineer (General) @ services]
   within-stratum doors: [('SAP Consultant', 0.53), ('Eng Manager / Tech Lead', 0.43), ('Frontend Developer', 0.39), ('Data Engineer', 0.35), ('Backend / Java Developer', 0.34), ('DevOps / SRE', 0.34), ('Full Stack Developer', 0.29), ('Solution / Tech Architect', 0.28), ('Data / BI Analyst', 0.26), ('Salesforce Developer', 0.25)]
   cross-stratum (GCC, labelled — may need credential/brand barriers): [('Software Engineer (General)', 0.31), ('Backend / Java Developer', 0.25)]
[Full Stack Developer @ services]
   within-stratum doors: [('Backend / Java Developer', 0.54), ('Frontend Developer', 0.47), ('.NET De

In [10]:
# --- moat headline: roles splittable into BOTH strata (compute, don't inherit Step-1's "13") ---
# A role is splittable iff it has >= MIN_STRATUM postings in BOTH services AND gcc.
# Counted over graph-eligible roles (node_roles) so it matches the graph population.
def _strat_n(role, et):
    return int(((sub.role == role) & (sub.emp_type == et)).sum())

both = [r for r in node_roles
        if _strat_n(r, 'services') >= MIN_STRATUM and _strat_n(r, 'gcc') >= MIN_STRATUM]
services_only = [r for r in node_roles
                 if _strat_n(r, 'services') >= MIN_STRATUM and _strat_n(r, 'gcc') < MIN_STRATUM]
gcc_only = [r for r in node_roles
            if _strat_n(r, 'services') < MIN_STRATUM and _strat_n(r, 'gcc') >= MIN_STRATUM]

print(f"MIN_STRATUM = {MIN_STRATUM}")
print(f"splittable into BOTH strata (services AND gcc >= {MIN_STRATUM}): {len(both)}")
for r in sorted(both):
    print(f"   {r:30s} services={_strat_n(r,'services'):5d}  gcc={_strat_n(r,'gcc'):5d}")
print(f"services-node only: {len(services_only)} | gcc-node only: {len(gcc_only)}")
print(f">> this is the verified moat number — replace the inherited '13' in the context doc with len(both).")


MIN_STRATUM = 25
splittable into BOTH strata (services AND gcc >= 25): 13
   Backend / Java Developer       services=  426  gcc=   56
   Data / BI Analyst              services=   74  gcc=   55
   Data Engineer                  services=  298  gcc=   45
   Data Scientist / ML            services=  247  gcc=   65
   DevOps / SRE                   services=  240  gcc=   61
   Eng Manager / Tech Lead        services=  296  gcc=   29
   Frontend Developer             services=  312  gcc=   35
   Full Stack Developer           services=  320  gcc=   47
   QA / Test Engineer             services=  210  gcc=   43
   SAP Consultant                 services=  795  gcc=   47
   Security Engineer              services=   84  gcc=   25
   Software Engineer (General)    services= 1415  gcc=  203
   Solution / Tech Architect      services=  114  gcc=   30
services-node only: 5 | gcc-node only: 0
>> this is the verified moat number — replace the inherited '13' in the context doc with len(both).


## 8 · Fix C — two-hop Pathfinder (explicit intermediate)

Every role gets a *path*, not a verdict. **Same-stratum intermediates in the default view**; cross-stratum stepping-stones are surfaced only as a labelled overlay door. Two modes, applied uniformly (no "island" label anywhere):
- **Has direct doors** → show them + the single neighbour that *opens the most additional roles* (the high-leverage onward door) and what it opens.
- **No direct doors** → name the *nearest* role (the stretch door), the skills to bridge to it, and what reaching it opens.

Run on the **services stratum** (the locked default view).

In [11]:
# ---- Fix C — two-hop Pathfinder with SPECIFICITY-WEIGHTED onward doors ----
# Onward door scored by (new_roles_opened * distinctiveness). Distinctiveness is
# measured as cross-role RARITY: a hub's top skills (python/sql/c#) appear in
# almost every role -> low; a specialist's (kubernetes/sap abap/selenium) are
# rare -> high. Measured against the graph, not within the node.

serv = [n for n in snodes if s_strat(n) == 'services']

# document frequency of each skill across role cores (how many roles share it)
_role_df = Counter()
for r in node_roles:
    for sk in set(core(r, k=30)):          # wider core so rare skills are counted
        _role_df[sk] += 1
_n_roles = len(node_roles)

def distinctiveness(node):
    skills = core(s_role(node), k=12, stratum='services')
    if not skills: return 0.0
    rarity = [ _n_roles / _role_df.get(sk, 1) for sk in skills ]
    return float(np.mean(rarity))          # ~1 = universal skills (hub); higher = rarer (specialist)

DISTINCT_FLOOR = 5.5   # READ the d= spread first, THEN set between hub and specialists

def sim(a, b): return float(SSIM[snodes.index(a)][snodes.index(b)])
def direct(n, thr): return sorted([m for m in serv if m != n and sim(n, m) >= thr], key=lambda m: -sim(n, m))

def gap_to(src_role, dst_role):
    have = set(core(src_role, stratum='services'))
    return [sk for sk in core(dst_role, stratum='services') if sk not in have][:5]

def score_onward(n, thr):
    """Rank candidate onward doors by new-roles-opened * distinctiveness."""
    d = direct(n, thr); dset = set(d) | {n}
    cands = []
    for m in d:
        new = {x for x in direct(m, thr) if x not in dset}
        dist = distinctiveness(m)
        cands.append((m, len(new), dist, len(new) * dist, sorted(s_role(x) for x in new)))
    return sorted(cands, key=lambda c: -c[3])   # by weighted score

def pathfinder(n, thr):
    rname = s_role(n); d = direct(n, thr)
    print(f"[{rname}]  (services)")
    if not d:
        # genuinely sparse within stratum — nearest stretch door + skills to bridge
        nearest = max((m for m in serv if m != n), key=lambda m: sim(n, m))
        print(f"   no direct door. nearest door: '{s_role(nearest)}' (cos {sim(n, nearest):.2f}, a stretch)")
        print(f"      skills to bridge there: {gap_to(rname, s_role(nearest))}")
        print(f"      reaching it opens ({len(direct(nearest, thr))}): {[s_role(m) for m in direct(nearest, thr)]}")
        return
    print(f"   reachable now ({len(d)}): {[s_role(m) for m in d]}")
    ranked = [c for c in score_onward(n, thr) if c[1] > 0]   # must open >=1 new role
    if not ranked:
        print("   no onward door opens new roles — your direct doors above are the move.")
        return
    # show top-2 candidates so the weighting is auditable
    print("   onward-door candidates (door | +new | distinct | score):")
    for m, nnew, dist, sc, _ in ranked[:2]:
        print(f"      {s_role(m):28s} +{nnew}  d={dist:.2f}  score={sc:.2f}")
    best = ranked[0]; m, nnew, dist, sc, newroles = best
    if DISTINCT_FLOOR is None or dist >= DISTINCT_FLOOR:
        print(f"   >> highest-leverage onward door: '{s_role(m)}' "
              f"(learn {gap_to(rname, s_role(m))}) -> opens +{nnew}: {newroles}")
    else:
        # combined a+b: name the weakness, still give a direction
        print(f"   >> no standout next specialism from here — your direct doors are the real move.")
        print(f"      if pushing further: '{s_role(m)}' (best available, but hub-like d={dist:.2f}) "
              f"-> opens +{nnew}: {newroles}")

for n in serv:
    pathfinder(n, GRAPH_THRESHOLD); print()

[SAP Consultant]  (services)
   reachable now (3): ['Software Engineer (General)', 'Solution / Tech Architect', 'Eng Manager / Tech Lead']
   onward-door candidates (door | +new | distinct | score):
      Eng Manager / Tech Lead      +4  d=11.11  score=44.45
      Software Engineer (General)  +7  d=4.81  score=33.64
   >> highest-leverage onward door: 'Eng Manager / Tech Lead' (learn ['c#', 'technical leadership', 'python', 'software development methodologies', 'c++']) -> opens +4: ['Backend / Java Developer', 'DevOps / SRE', 'Frontend Developer', 'Full Stack Developer']

[Software Engineer (General)]  (services)
   reachable now (10): ['SAP Consultant', 'Eng Manager / Tech Lead', 'Frontend Developer', 'Data Engineer', 'Backend / Java Developer', 'DevOps / SRE', 'Full Stack Developer', 'Solution / Tech Architect', 'Data / BI Analyst', 'Salesforce Developer']
   onward-door candidates (door | +new | distinct | score):
      Data Engineer                +2  d=11.37  score=22.74
      Dev

## 9 · Diagnostic 2 — de-hub on/off (does `Software Engineer (General)` act as a connective bridge?)

The independent review flagged that re-routing the generic node might artificially *lengthen* paths if it was a real connector. This recomputes isolated-node count with the generic bucket left intact, on the full role graph at the working threshold.

In [12]:
sub2 = sub.copy()
sub2["role"] = sub2["title"].apply(assign_role)   # NO de-hub
rc2 = sub2["role"].value_counts()
nr2 = [r for r,c in rc2.items() if r!='unclassified' and c>=MIN_ROLE_POSTINGS]
rd2 = {r:[s for lst in sub2.loc[sub2.role==r,"skills"] for s in lst] for r in nr2}
v2 = TfidfVectorizer(analyzer=lambda x:x, min_df=2, sublinear_tf=True)
X2 = v2.fit_transform([rd2[r] for r in nr2]); S2 = cosine_similarity(X2)
def iso_count(sim_mat, names, thr):
    G=nx.Graph(); G.add_nodes_from(names)
    for i in range(len(names)):
        for j in range(i+1,len(names)):
            if sim_mat[i][j]>=thr: G.add_edge(names[i],names[j])
    return [n for n in names if G.degree(n)==0]
iso_on  = iso_count(SIM, roles, GRAPH_THRESHOLD)
iso_off = iso_count(S2, nr2, GRAPH_THRESHOLD)
print(f"isolated WITH de-hub    ({len(iso_on)}): {iso_on}")
print(f"isolated WITHOUT de-hub ({len(iso_off)}): {iso_off}")
print(">> if WITHOUT has fewer isolated, the generic node was a real bridge and de-hub over-corrected.")


isolated WITH de-hub    (0): []
isolated WITHOUT de-hub (0): []
>> if WITHOUT has fewer isolated, the generic node was a real bridge and de-hub over-corrected.


## 10 · Verdict — paste back

- **Diagnostic 1** (cleaned Gate-1 NN) — confirm the validated connections survived cleaning.
- **Fix A** (threshold sweep + isolated-node count) — how many of Step-1's 7 islands survive the correct metric.
- **Fix B** (within-stratum doors + cross-stratum overlay) — does the moat hold with real cross-stratum doors surfaced honestly.
- **Fix C** (Pathfinder) — does every role get a defensible path (direct doors, or nearest door + bridge skills).
- **Diagnostic 2** — did de-hub over-correct.

Decision: pick the working `GRAPH_THRESHOLD` from the sweep, then read whether the genuinely-sparse roles (likely Security / Network) are few and honestly handled by the nearest-door Pathfinder, vs metric-starved roles that reconnected.
